# NB-14 — Rebuilding Table 4: gating-only and modulation-only controls, exact costs (WP11, WP17)

**Why.** Table 4 compared specialised fusion front ends with masked concatenation, but (i) the decisive rows had **one seed**, (ii) the gated U-Net was trained **with** the consistency loss while the concat U-Net row was trained **without** it (the same objective mismatch we found in Table 2), (iii) "equal parameter budget" was stated but never measured, and (iv) there was no gating-only or modulation-only control. This notebook fixes all four, under the identical Study-1 protocol (Focal-Tversky, AdamW 3e-4, wd 1e-4, 45 epochs, batch 8, EMA 0.99, grad-clip 1, seeds 42 / 1 / 2, selection on mean val IoU over the 3 regimes).

| Group | Runs (× 3 seeds) | What it separates |
|---|---|---|
| **A_unet_frontends** (12 runs) | `unet_gate_film` (the paper's gated variant), `unet_gate_only`, `unet_film_only`, `unet_gate_film_nocons` (gated, consistency 0) | gate vs modulation vs both on the U-Net; partner for the "consistency removed" row. Partners: `unet_full` (NB-13, concat, curriculum + consistency) and `unet_drop` (NB-11, concat, curriculum only) |
| **B_segformer_frontends** (9 runs) | `seg_gate_film`, `seg_gate_only`, `seg_film_only` | same on SegFormer-B0. Partner: `seg_full_ref` (NB-12, concat, curriculum + consistency) |
| *C_adapter_controls* (6 runs, optional, off by default) | `unet_adapter_only`, `seg_adapter_only`: the front end with **neither** gate nor FiLM (adapters + 1×1 projection only) | how much of any change comes from the extra front-end layers themselves |

Also in this notebook: the **exact parameter count of every component**, FLOPs (with the fixed `flops_g`), latency and peak memory on the named GPU, and the measured training minutes per run (including the cost of the consistency pass).

**Rules fixed before looking at the results**
1. A fusion mechanism "helps" (or "hurts") only if the difference to the matching concat model on **test SAR-only IoU** exceeds 0.03 **and** has the same sign in all 3 seeds. Otherwise the paper says "no detectable effect". The same rule is printed for the other two regimes.
2. "Equal parameter budget" is not claimed. The notebook prints the exact counts and the paper reports them.
3. Bolivia is reported but not used to decide.
4. Every comparison is between models trained with the **same objective** (concat and front-end both with, or both without, the consistency loss).

**Tuning budget for these runs: none.** One configuration per variant, the same hyper-parameters as the concat model, no search.

**Attach** (Add Input): `lgmf_v8` · Sen1Floods11 · `lgmf-index` · the **NB-13**, **NB-12** and **NB-11** notebook outputs · on later commits, the previous commit of this notebook.
**Settings:** GPU T4 · Internet **On** · Save & Run All (Commit). Expect about 14 min per run (gated front ends run at full resolution), so 21 runs ≈ 5 h. The loop stops by itself before the session limit; attach the output to a new commit and run again.

## 0. Bootstrap

In [ ]:
import sys, os, glob, json, time, shutil, warnings, re, importlib, dataclasses, copy
warnings.filterwarnings("ignore")
T_START = time.time()
for _m in [m for m in sys.modules if m == "lgmf" or m.startswith("lgmf.")]: del sys.modules[_m]
importlib.invalidate_caches()
REQUIRED = ["__init__.py", "config.py", "dataset.py", "model.py", "baselines.py", "engine.py", "curriculum.py", "metrics.py",
            "losses.py", "ema.py", "cloud.py", "ramdata.py", "audit.py", "runner.py", "selftest_v8.py"]
def _ver(p):
    m = re.search(r'__version__\s*=\s*"([\d.]+)"', open(p).read())
    return tuple(int(x) for x in m.group(1).split(".")) if m else (0,)
_cands = []
print("every lgmf/ folder found under /kaggle/input:")
for p in sorted(glob.glob("/kaggle/input/**/lgmf/__init__.py", recursive=True)):
    d = os.path.dirname(p); miss = [f for f in REQUIRED if not os.path.exists(os.path.join(d, f))]
    print(f"   v{'.'.join(map(str, _ver(p)))}  {'COMPLETE' if not miss else 'missing ' + ','.join(miss)}  {d}")
    if not miss: _cands.append((_ver(p), d))
assert _cands, "attach the lgmf_v8 dataset (v7 and older do not have runner/ramdata/cloud/audit)"
_best = max(_cands); assert _best[0] >= (8,), f"best complete lgmf is v{_best[0]}; attach lgmf_v8"
PKG_DIR = _best[1]; sys.path = [os.path.dirname(PKG_DIR)] + [q for q in sys.path if q != os.path.dirname(PKG_DIR)]
for mod, pipname in (("segmentation_models_pytorch", "segmentation-models-pytorch"), ("thop", "thop")):
    try: __import__(mod)
    except ImportError: os.system(f"pip install -q {pipname}")
import numpy as np, pandas as pd, torch
import lgmf
assert os.path.dirname(lgmf.__file__) == PKG_DIR, lgmf.__file__
from lgmf.config import LGMFConfig
from lgmf.curriculum import regime_probs
from lgmf.ramdata import RamDataset
from lgmf.runner import run_experiment, is_done, run_dir, build_from_spec
from lgmf.audit import metrics_from_counts
from lgmf.model import count_params
from lgmf.metrics import flops_g
print("lgmf", lgmf.__version__, "|", PKG_DIR)

ns = sorted(glob.glob("/kaggle/input/**/norm_stats.json", recursive=True)); assert ns, "attach lgmf-index"
idx = os.path.dirname(ns[0]); stats = json.load(open(ns[0]))
for s in ("train", "val", "test", "bolivia"): assert os.path.exists(f"{idx}/{s}_index.json"), s

# --- the index files store absolute image paths from the session that built them. If this session mounts
# --- Sen1Floods11 somewhere else, re-point every path by file name (names are unique) into /kaggle/working/index_fixed.
def _paths_ok(d):
    it = json.load(open(f"{d}/train_index.json"))[0]
    return all(os.path.exists(it[k]) for k in ("s1", "s2", "label") if it.get(k))
if not _paths_ok(idx):
    hits = glob.glob("/kaggle/input/**/HandLabeled", recursive=True)
    assert hits, "index paths are invalid and no 'HandLabeled' folder was found: attach the Sen1Floods11 dataset"
    where = {f: os.path.join(r, f) for r, _, fs in os.walk(hits[0]) for f in fs if f.endswith(".tif")}
    FIX = "/kaggle/working/index_fixed"; os.makedirs(FIX, exist_ok=True); lost = []
    for s in ("train", "val", "test", "bolivia"):
        items = json.load(open(f"{idx}/{s}_index.json"))
        for it in items:
            for k in ("s1", "s2", "label", "jrc"):
                if it.get(k):
                    nm = os.path.basename(it[k])
                    if nm in where: it[k] = where[nm]
                    else: lost.append((s, k, nm))
        json.dump(items, open(f"{FIX}/{s}_index.json", "w"))
    assert not lost, f"{len(lost)} image files not found, e.g. {lost[:3]}"
    print(f"index paths re-pointed to {hits[0]}  ->  {FIX}"); idx = FIX
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "(CPU: only for the smoke test)")

# ------------------------------ the knobs ------------------------------
EPOCHS = int(os.environ.get("NB14_EPOCHS", 45))        # never change mid-stream: results live in runs_e<EPOCHS>
SEEDS = [42, 1, 2]
GROUPS_TO_RUN = ["A_unet_frontends", "B_segformer_frontends"]    # add "C_adapter_controls" for the optional control
TIME_BUDGET_H = 10.5
OUT = "/kaggle/working"
MY_IDS = {"unet_gate_film", "unet_gate_only", "unet_film_only", "unet_gate_film_nocons", "seg_gate_film", "seg_gate_only", "seg_film_only",
          "unet_adapter_only", "seg_adapter_only"}

## 1. Preflight (about 1 min)

In [ ]:
from lgmf import selftest_v8
selftest_v8.run(device=device)

## 2. RESTORE — carry finished NB-14 runs forward (only this notebook's own run ids; NB-11/12/13 outputs are references, never copied)

In [ ]:
def restore_previous():
    n = 0
    for p in glob.glob(f"/kaggle/input/**/runs_e{EPOCHS}/*/seed*/result.json", recursive=True):
        rid = p.split(os.sep)[-3]
        if rid not in MY_IDS: continue
        src = os.path.dirname(p); rel = src.split(f"runs_e{EPOCHS}" + os.sep, 1)[1]
        dst = os.path.join(OUT, f"runs_e{EPOCHS}", rel)
        if not os.path.exists(os.path.join(dst, "result.json")):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            if os.path.exists(dst): shutil.rmtree(dst)
            shutil.copytree(src, dst); n += 1
    return n
print("restored", restore_previous(), "finished NB-14 run(s) from attached inputs")
for p in sorted(glob.glob(f"{OUT}/runs_e{EPOCHS}/*/seed*/result.json")): print("  have", p.split(f"runs_e{EPOCHS}/")[1].rsplit("/result.json", 1)[0])

## 3. Data into RAM (once) and the run table

In [ ]:
base = LGMFConfig(); base.s1_mean, base.s1_std = stats["s1_mean"], stats["s1_std"]; base.s2_mean, base.s2_std = stats["s2_mean"], stats["s2_std"]
t0 = time.time()
TRAIN = RamDataset(base, f"{idx}/train_index.json", augment=True)
EVAL = {s: RamDataset(base, f"{idx}/{s}_index.json", augment=False) for s in ("val", "test", "bolivia")}
print({"train": len(TRAIN), **{k: len(v) for k, v in EVAL.items()}}, f"cached in {time.time()-t0:.0f}s")
assert len(TRAIN) == 250 and len(EVAL["val"]) == 89 and len(EVAL["test"]) == 90 and len(EVAL["bolivia"]) == 15, "split sizes differ from the paper"

def make_cfg(overrides, seed):
    c = LGMFConfig()
    c.epochs, c.lr, c.batch_size, c.ema_decay = EPOCHS, 3e-4, 8, 0.99
    c.loss_type, c.consistency_weight, c.num_workers = "focal_tversky", 0.5, 0
    c.s1_mean, c.s1_std, c.s2_mean, c.s2_std = stats["s1_mean"], stats["s1_std"], stats["s2_mean"], stats["s2_std"]
    c.seed = seed
    for k, v in overrides.items(): setattr(c, k, v)
    return c

SEG   = dict(kind="concat", arch="segformer", encoder="mit_b0")
UNET  = dict(kind="concat", arch="unet",  encoder="mobilenet_v2")
MANET = dict(kind="concat", arch="manet", encoder="mobilenet_v2")
def GU(arch, enc, film, gate): return dict(kind="gated", arch=arch, encoder=enc, film=film, gate=gate)
U, S = ("unet", "mobilenet_v2"), ("segformer", "mit_b0")
RUNS = [   # (run_id, group, model spec, config overrides)    selection is always the mean val IoU over the 3 regimes
    ("unet_gate_film",       "A_unet_frontends",      GU(*U, True,  True),  {}),
    ("unet_gate_only",       "A_unet_frontends",      GU(*U, False, True),  {}),
    ("unet_film_only",       "A_unet_frontends",      GU(*U, True,  False), {}),
    ("unet_gate_film_nocons","A_unet_frontends",      GU(*U, True,  True),  dict(consistency_weight=0.0)),
    ("seg_gate_film",        "B_segformer_frontends", GU(*S, True,  True),  {}),
    ("seg_gate_only",        "B_segformer_frontends", GU(*S, False, True),  {}),
    ("seg_film_only",        "B_segformer_frontends", GU(*S, True,  False), {}),
    ("unet_adapter_only",    "C_adapter_controls",    GU(*U, False, False), {}),
    ("seg_adapter_only",     "C_adapter_controls",    GU(*S, False, False), {}),
]
assert {r[0] for r in RUNS} == MY_IDS
display(pd.DataFrame([dict(run=r, group=g, backbone=s["arch"], gate=s["gate"], film=s["film"], consistency=o.get("consistency_weight", 0.5)) for r, g, s, o in RUNS]))

## 4. Train (seed-major; resumable; time estimate kept per architecture; stops before the session limit)

In [ ]:
todo = [(rid, g, spec, ov, sd) for sd in SEEDS for rid, g, spec, ov in RUNS if g in GROUPS_TO_RUN and not is_done(OUT, EPOCHS, rid, sd)]
print(f"{len(todo)} run(s) to do in {GROUPS_TO_RUN}")
dur = {}
for rid, g, spec, ov, sd in todo:
    seen = [x for v in dur.values() for x in v]
    est = (np.mean(dur[spec["arch"]]) if spec["arch"] in dur else (max(seen) * 1.5 if seen else 35.0)) * 1.15
    left = TIME_BUDGET_H * 60 - (time.time() - T_START) / 60
    if est > left:
        print(f"STOP: next run ({rid}, seed {sd}) needs ~{est:.0f} min, only {left:.0f} min of budget left. "
              f"Save this version, attach its output to a new commit, and run again (finished runs are skipped).")
        break
    t1 = time.time()
    run_experiment(rid, make_cfg(ov, sd), spec, TRAIN, EVAL["val"], EVAL, OUT, device, select="mean")
    dur.setdefault(spec["arch"], []).append((time.time() - t1) / 60)
print("minutes per run by architecture:", {k: round(float(np.mean(v)), 1) for k, v in dur.items()})
print(f"done; {sum(len(v) for v in dur.values())} run(s) this session, {(time.time()-T_START)/3600:.1f} h elapsed")

## 5. Results — every run on val, test and Bolivia, mean ± std over seeds (ddof = 1)
Row prefixes: no prefix = trained here · `prev:` = same-protocol runs from the attached NB-12 / NB-13 outputs (`prev:unet_full`, `prev:seg_full_ref`, …) · `ref11:` = the earlier runs behind the paper (NB-11 audit). `ref11:unet_drop` and `ref11:segformer_noconsist` had **consistency weight 0**; `ref11:segformer_gated` is the paper's SegFormer gated model.

In [ ]:
recs, seen_ref = [], set()
def add_result(r, prefix=""):
    for split in ("val", "test", "bolivia"):
        for reg, m in r[split].items():
            recs.append(dict(run=prefix + r["run_id"], seed=r["seed"], split=split, regime=reg, iou=m["iou"], flood_iou=m["flood_iou"]))
TRAIN_MIN = {}
for p in glob.glob(f"{OUT}/runs_e{EPOCHS}/*/seed*/result.json"):
    r = json.load(open(p)); add_result(r); TRAIN_MIN[(r["run_id"], r["seed"])] = r["train_minutes"]
for p in glob.glob(f"/kaggle/input/**/runs_e{EPOCHS}/*/seed*/result.json", recursive=True):
    r = json.load(open(p))
    if r["run_id"] in MY_IDS or (r["run_id"], r["seed"]) in seen_ref: continue
    seen_ref.add((r["run_id"], r["seed"])); add_result(r, "prev:"); TRAIN_MIN[("prev:" + r["run_id"], r["seed"])] = r["train_minutes"]
print("same-protocol reference runs found (NB-12 / NB-13):", sorted({k[0] for k in seen_ref}) or "NONE (attach the NB-12 and NB-13 outputs)")
ref_files = glob.glob("/kaggle/input/**/per_chip_counts.csv.gz", recursive=True)
if ref_files:
    pc = pd.read_csv(ref_files[0]); pc = pc[(pc.variant == "zero") & pc.config.isin(["segformer_concat", "segformer_noconsist", "segformer_gated", "unet_drop", "manet_drop", "unet_naive"])]
    g = pc.groupby(["config", "seed", "split", "regime"])[["tp", "fp", "fn", "tn", "ftp", "ffp", "ffn", "ftn"]].sum().reset_index()
    g["iou"] = metrics_from_counts(g.tp, g.fp, g.fn, g.tn)["iou"]; g["flood_iou"] = metrics_from_counts(g.ftp, g.ffp, g.ffn, g.ftn)["iou"]
    for r in g.itertuples(): recs.append(dict(run="ref11:" + r.config, seed=int(r.seed), split=r.split, regime=r.regime, iou=r.iou, flood_iou=r.flood_iou))
    print("NB-11 references loaded from", ref_files[0])
else:
    print("NB-11 output not attached: no ref11:* rows")
R = pd.DataFrame(recs); R.to_csv(f"{OUT}/nb14_per_seed.csv", index=False)

ORDER = ["ref11:unet_drop", "prev:unet_full", "unet_gate_film_nocons", "unet_gate_film", "unet_gate_only", "unet_film_only", "unet_adapter_only",
         "ref11:segformer_noconsist", "prev:seg_full_ref", "ref11:segformer_gated", "seg_gate_film", "seg_gate_only", "seg_film_only", "seg_adapter_only"]
def table(split):
    g = R[R.split == split].groupby(["run", "regime"]).iou.agg(["mean", "std", "count"]).reset_index()
    g["cell"] = g.apply(lambda r: f"{r['mean']:.3f}±{(r['std'] if r['std'] == r['std'] else 0):.3f}", axis=1)
    t = g.pivot(index="run", columns="regime", values="cell")[["both", "s1_only", "s2_only"]]
    t["n"] = g.groupby("run")["count"].max()
    keep = [x for x in ORDER if x in t.index]
    return t.reindex(keep)
pd.set_option("display.width", 250)
for sp in ("val", "test", "bolivia"):
    print(f"\n=== {sp.upper()}  (IoU, mean ± std over seeds, ddof=1) ==="); print(table(sp).to_string())
    table(sp).to_csv(f"{OUT}/nb14_table_{sp}.csv")

## 6. Paired contrasts (same seed) and the pre-set classification
Each front-end variant is compared with the concat model of the **same backbone and the same objective**. `same` = identical sign in all seeds.

In [ ]:
def per_seed(run, split, regime):
    return R[(R.run == run) & (R.split == split) & (R.regime == regime)].set_index("seed")["iou"]
def pair(A, B, split, reg):
    a, b = per_seed(A, split, reg), per_seed(B, split, reg); c = a.index.intersection(b.index)
    return (a[c] - b[c]) if len(c) else None
CON = []   # (question, label, A, B) -> A minus B
for bb, concat, pre, gf, go, fo, ao in (("U-Net", "prev:unet_full", "unet", "unet_gate_film", "unet_gate_only", "unet_film_only", "unet_adapter_only"),
                                         ("SegFormer", "prev:seg_full_ref", "seg", "seg_gate_film", "seg_gate_only", "seg_film_only", "seg_adapter_only")):
    CON += [(bb, "gate + FiLM minus concat", gf, concat), (bb, "gate-only minus concat", go, concat), (bb, "FiLM-only minus concat", fo, concat),
            (bb, "FiLM effect given the gate (gate+FiLM minus gate-only)", gf, go), (bb, "gate effect given FiLM (gate+FiLM minus FiLM-only)", gf, fo),
            (bb, "adapter-only minus concat (front-end layers alone)", ao, concat), (bb, "gate-only minus adapter-only", go, ao), (bb, "FiLM-only minus adapter-only", fo, ao)]
CON += [("U-Net, NO consistency", "gate + FiLM minus concat, both without consistency", "unet_gate_film_nocons", "ref11:unet_drop"),
        ("U-Net, consistency effect", "removing consistency from the gated model (nocons minus with)", "unet_gate_film_nocons", "unet_gate_film"),
        ("U-Net, consistency effect", "removing consistency from the concat model (unet_drop minus unet_full)", "ref11:unet_drop", "prev:unet_full")]
rows, last = [], None
for q, lab, A, B in CON:
    if q != last: print(f"\n##### {q}"); last = q
    print(f"  {lab}")
    for reg in ("s1_only", "s2_only", "both"):
        seg = []
        for sp in ("val", "test", "bolivia"):
            d = pair(A, B, sp, reg)
            if d is None: seg.append(f"{sp}: n/a"); continue
            same = bool((d > 0).all() or (d < 0).all())
            rows.append(dict(question=q, contrast=lab, split=sp, regime=reg, n=len(d), mean_diff=d.mean(), seed_sd=d.std() if len(d) > 1 else np.nan,
                             per_seed=" ".join(f"{s}:{v:+.3f}" for s, v in d.items()), same_sign=same))
            seg.append(f"{sp}: {d.mean():+.3f} [{' '.join(f'{s}:{v:+.3f}' for s, v in d.items())}]{' same' if same else ''}")
        print(f"      {reg:8s} " + "   ".join(seg))
C = pd.DataFrame(rows); C.to_csv(f"{OUT}/nb14_contrasts.csv", index=False)

def classify(d):
    if d is None or len(d) < 3: return "n/a (needs 3 seeds)"
    same = bool((d > 0).all() or (d < 0).all())
    return ("helps" if d.mean() > 0 else "hurts") if (same and abs(d.mean()) > 0.03) else "no detectable effect"
print("\n=== Rule 1 (test split): helps / hurts only if |mean diff| > 0.03 AND same sign in all seeds ===")
out = []
for bb, concat, names in (("U-Net", "prev:unet_full", ("unet_gate_film", "unet_gate_only", "unet_film_only", "unet_adapter_only")),
                          ("SegFormer", "prev:seg_full_ref", ("seg_gate_film", "seg_gate_only", "seg_film_only", "seg_adapter_only"))):
    for n in names:
        row = dict(backbone=bb, variant=n)
        for reg in ("s1_only", "s2_only", "both"):
            d = pair(n, concat, "test", reg); row[reg] = "n/a" if d is None else f"{d.mean():+.3f}  {classify(d)}"
        out.append(row)
print(pd.DataFrame(out).to_string(index=False)); pd.DataFrame(out).to_csv(f"{OUT}/nb14_rule1_classification.csv", index=False)

## 7. Component card — what exactly each front end contains (read from the model objects, not typed by hand)

In [ ]:
def nparams(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
torch.manual_seed(0)
rows = []
for name, spec in (("U-Net gate+FiLM", GU(*U, True, True)), ("U-Net gate-only", GU(*U, False, True)), ("U-Net FiLM-only", GU(*U, True, False)), ("U-Net adapter-only", GU(*U, False, False)),
                   ("SegFormer gate+FiLM", GU(*S, True, True)), ("SegFormer gate-only", GU(*S, False, True)), ("SegFormer FiLM-only", GU(*S, True, False)), ("SegFormer adapter-only", GU(*S, False, False))):
    m = build_from_spec(spec, pretrained=False); f = m.front
    comp = dict(adapt_S1=nparams(f.adapt1), adapt_S2=nparams(f.adapt2), gate_MLP=nparams(f.gate_mlp) if getattr(f, "use_gate", False) else 0,
                FiLM=nparams(f.film) if f.film is not None else 0, proj_1x1=nparams(f.proj))
    rows.append(dict(model=name, backbone_params=nparams(m.net), front_end_params=nparams(f), **comp, total_params=nparams(m)))
comp_df = pd.DataFrame(rows)
for n, spec in (("U-Net concat", UNET), ("SegFormer concat", SEG), ("MANet concat", MANET)):
    m = build_from_spec(spec, pretrained=False)
    comp_df = pd.concat([comp_df, pd.DataFrame([dict(model=n, backbone_params=nparams(m), front_end_params=0, total_params=nparams(m))])], ignore_index=True)
print(comp_df.fillna(0).to_string(index=False))
comp_df.to_csv(f"{OUT}/nb14_component_params.csv", index=False)
base_p = {"U-Net": float(comp_df[comp_df.model == "U-Net concat"].total_params.iloc[0]), "SegFormer": float(comp_df[comp_df.model == "SegFormer concat"].total_params.iloc[0])}
print("\nextra parameters of the gate+FiLM front end vs masked concatenation:",
      {"U-Net": f"{float(comp_df[comp_df.model == 'U-Net gate+FiLM'].total_params.iloc[0]) - base_p['U-Net']:+,.0f} ({100*(float(comp_df[comp_df.model == 'U-Net gate+FiLM'].total_params.iloc[0]) / base_p['U-Net'] - 1):+.2f} %)",
       "SegFormer": f"{float(comp_df[comp_df.model == 'SegFormer gate+FiLM'].total_params.iloc[0]) - base_p['SegFormer']:+,.0f} ({100*(float(comp_df[comp_df.model == 'SegFormer gate+FiLM'].total_params.iloc[0]) / base_p['SegFormer'] - 1):+.2f} %)"})

# initialisation actually present in the code: PyTorch defaults; FiLM is NOT zero-initialised
m = build_from_spec(GU(*U, True, True), pretrained=False).front.eval()
with torch.no_grad():
    gb = m.film.mlp(torch.tensor([[1., 1.], [1., 0.], [0., 1.]]))
    d = m.film.dim
    print("\nFiLM at initialisation (present = both / SAR-only / optical-only): mean |gamma| =", [round(float(gb[i, :d].abs().mean()), 3) for i in range(3)],
          "| mean |beta| =", [round(float(gb[i, d:].abs().mean()), 3) for i in range(3)],
          "\n  -> the code uses (1 + gamma) but the MLP is not zero-initialised, so the modulation is NOT the identity at the start; say 'default PyTorch initialisation'.")
print("\ncomponent definitions (from lgmf/baselines.py GatedFusionFrontEnd):")
print("  adapters: Conv3x3 + GroupNorm + ReLU per sensor (S1: 2->8 ch, S2: 6->24 ch), outputs multiplied by the availability flag")
print("  gate:     MLP(availability, pooled adapter features) -> sigmoid; the gate of an absent sensor is zeroed and the remaining gates are renormalised over the present sensors")
print("  FiLM:     MLP(availability) -> per-channel (gamma, beta); x * (1 + gamma) + beta on the concatenated 32 channels")
print("  proj:     1x1 conv 32 -> 8 channels, fed to the unchanged backbone (same in_channels as the concat model)")
print("  encoder:  ImageNet-pretrained, as in the concat models; all front-end layers: PyTorch default initialisation; tuning budget: none")

## 8. Cost on named hardware (WP17): parameters, GFLOPs (fixed `flops_g`), latency, peak memory, measured training time
One 512 × 512 chip, both sensors present, batch 1, eval mode, 10 warm-up + 50 timed iterations; all concat and gated models run the same computation in every availability regime (missing sensors are zero-filled, not skipped).

In [ ]:
import platform
hw = dict(gpu=torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU", torch=torch.__version__, cuda=torch.version.cuda, python=platform.python_version())
try:
    import segmentation_models_pytorch as _smp; hw["smp"] = _smp.__version__
except Exception: pass
print("hardware / software:", hw)
COST = {"SegFormer-B0 concat": SEG, "U-Net concat": UNET, "MANet concat": MANET,
        "SegFormer gate+FiLM": GU(*S, True, True), "SegFormer gate-only": GU(*S, False, True), "SegFormer FiLM-only": GU(*S, True, False), "SegFormer adapter-only": GU(*S, False, False),
        "U-Net gate+FiLM": GU(*U, True, True), "U-Net gate-only": GU(*U, False, True), "U-Net FiLM-only": GU(*U, True, False), "U-Net adapter-only": GU(*U, False, False)}
NIT, NW = (50, 10) if device.type == "cuda" else (3, 1)
def latency(m, amp):
    s1 = torch.randn(1, 2, 512, 512, device=device); s2 = torch.randn(1, 6, 512, 512, device=device); pr = torch.ones(1, 2, device=device)
    with torch.no_grad(), torch.autocast("cuda", enabled=amp and device.type == "cuda"):
        for _ in range(NW): m(s1, s2, pr)
        if device.type == "cuda": torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
        t = time.perf_counter()
        for _ in range(NIT): m(s1, s2, pr)
        if device.type == "cuda": torch.cuda.synchronize()
    return (time.perf_counter() - t) / NIT * 1000, (torch.cuda.max_memory_allocated() / 2**20 if device.type == "cuda" else float("nan"))
rows = []
for name, spec in COST.items():
    try:
        m = build_from_spec(spec, pretrained=False).to(device).eval()
        gf = flops_g(m, base, device="cpu")
        l32, mem32 = latency(m, False); l16, mem16 = latency(m, True)
        rows.append(dict(model=name, params_M=round(count_params(m), 3), GFLOPs=round(gf, 2), latency_fp32_ms=round(l32, 1), latency_amp_ms=round(l16, 1), peak_mem_fp32_MB=(round(mem32) if mem32 == mem32 else mem32), peak_mem_amp_MB=(round(mem16) if mem16 == mem16 else mem16)))
        del m
        if device.type == "cuda": torch.cuda.empty_cache()
    except Exception as e:
        rows.append(dict(model=name, params_M=np.nan, GFLOPs=np.nan)); print("cost failed for", name, "->", repr(e)[:150])
COSTDF = pd.DataFrame(rows); print(COSTDF.to_string(index=False)); COSTDF.to_csv(f"{OUT}/nb14_cost.csv", index=False)
print("(GFLOPs = -1 means the 'thop' package is missing: pip install thop and re-run this cell)")

# measured wall-clock training minutes per run (this notebook + the attached NB-12 / NB-13 outputs), same GPU type
tm = pd.DataFrame([dict(run=k[0], seed=k[1], minutes=v) for k, v in TRAIN_MIN.items()]).groupby("run").minutes.agg(["mean", "std", "count"]).round(1)
print("\nmeasured training minutes per run (45 epochs; includes per-epoch validation):"); print(tm.to_string())
if {"prev:seg_plain", "prev:seg_consonly"} <= set(tm.index):
    print(f"\nconsistency-pass overhead on SegFormer (same data, same GPU type): {tm.loc['prev:seg_consonly', 'mean']:.1f} min with vs {tm.loc['prev:seg_plain', 'mean']:.1f} min without "
          f"= x{tm.loc['prev:seg_consonly', 'mean'] / tm.loc['prev:seg_plain', 'mean']:.2f}")
tm.to_csv(f"{OUT}/nb14_train_minutes.csv")

## 9. Table 4, rebuilt
Block 1 = decisive rows: **3 seeds, v8 pipeline, identical objective for each pair** (validation and test shown; the original Table 4 printed validation numbers). Block 2 = the original single-seed exploratory rows, copied from the paper, **not re-run, support no claim**.

In [ ]:
def cell_(run, split, reg):
    v = R[(R.run == run) & (R.split == split) & (R.regime == reg)].iou.values
    return f"{v.mean():.3f}±{v.std(ddof=1):.3f}" if len(v) > 1 else (f"{v[0]:.3f}" if len(v) else "n/a")
def pcost(model):
    r = COSTDF[COSTDF.model == model]
    return (r.params_M.iloc[0], r.GFLOPs.iloc[0]) if len(r) else (np.nan, np.nan)
T4 = [("U-Net", "masked concatenation, curriculum + consistency", "prev:unet_full", "U-Net concat"),
      ("U-Net", "gate + FiLM (the paper's gated variant)", "unet_gate_film", "U-Net gate+FiLM"),
      ("U-Net", "gate only", "unet_gate_only", "U-Net gate-only"),
      ("U-Net", "FiLM only", "unet_film_only", "U-Net FiLM-only"),
      ("U-Net", "masked concatenation, curriculum only (no consistency)", "ref11:unet_drop", "U-Net concat"),
      ("U-Net", "gate + FiLM, no consistency", "unet_gate_film_nocons", "U-Net gate+FiLM"),
      ("SegFormer-B0", "masked concatenation, curriculum + consistency", "prev:seg_full_ref", "SegFormer-B0 concat"),
      ("SegFormer-B0", "gate + FiLM", "seg_gate_film", "SegFormer gate+FiLM"),
      ("SegFormer-B0", "gate only", "seg_gate_only", "SegFormer gate-only"),
      ("SegFormer-B0", "FiLM only", "seg_film_only", "SegFormer FiLM-only")]
rows = []
for bb, lab, run, cname in T4:
    p, g = pcost(cname)
    rows.append(dict(backbone=bb, fusion=lab, params_M=p, GFLOPs=g,
                     val_both=cell_(run, "val", "both"), val_SAR=cell_(run, "val", "s1_only"), val_opt=cell_(run, "val", "s2_only"),
                     test_both=cell_(run, "test", "both"), test_SAR=cell_(run, "test", "s1_only"), test_opt=cell_(run, "test", "s2_only"), bolivia_SAR=cell_(run, "bolivia", "s1_only"),
                     seeds=int(R[(R.run == run) & (R.split == "test") & (R.regime == "both")].shape[0])))
T4DF = pd.DataFrame(rows); print(T4DF.to_string(index=False)); T4DF.to_csv(f"{OUT}/nb14_table4_rebuilt.csv", index=False)
print("\nExploratory rows from the original Table 4 (validation, single seed unless stated, NOT re-run, support no claim):")
print(pd.DataFrame([dict(block="Shared encoder, modality-specific stems", row="lightweight decoder, gate + FiLM", both=0.685, SAR=0.531, opt=0.607, seeds=1),
                    dict(block="Shared encoder, modality-specific stems", row="skip-connection decoder, gate + FiLM", both=0.724, SAR=0.535, opt=0.653, seeds=1),
                    dict(block="U-Net (paper)", row="gate + FiLM", both=0.780, SAR=0.598, opt=0.715, seeds=1),
                    dict(block="U-Net (paper)", row="gate, no consistency", both=0.772, SAR=0.510, opt=0.735, seeds=1),
                    dict(block="SegFormer (paper)", row="gate + FiLM, 3 seeds, earlier protocol", both=0.755, SAR=0.557, opt=0.735, seeds=3)]).to_string(index=False))

## 10. Measured regime exposure of this notebook's runs

In [ ]:
rows = []
for p in sorted(glob.glob(f"{OUT}/runs_e{EPOCHS}/*/seed*/result.json")):
    r = json.load(open(p))
    rows.append(dict(run=r["run_id"], seed=r["seed"], measured_both=r["measured_cumulative_shares"][0], measured_sar=r["measured_cumulative_shares"][1],
                     measured_opt=r["measured_cumulative_shares"][2], scheduled_both=r["scheduled_cumulative_probs"][0], scheduled_sar=r["scheduled_cumulative_probs"][1],
                     scheduled_opt=r["scheduled_cumulative_probs"][2], best_epoch=r["best_epoch"], params_M=r["params_M"], train_min=r["train_minutes"]))
X = pd.DataFrame(rows)
if len(X):
    g = X.groupby("run").mean(numeric_only=True).drop(columns="seed"); print(g.round(4).to_string()); g.to_csv(f"{OUT}/nb14_measured_exposure.csv")

## 11. What to send back
When the commit has finished: send me the printed output of sections 5–10 (or the executed `.ipynb`). If the loop printed `STOP: …`, save the version, attach its output to a new commit of this notebook and run again.
Optional group C (`"C_adapter_controls"`) only if the gated variants differ from concat and you want to know whether the extra front-end layers alone explain it.